# 03 – Entrenamiento Modelo 2 (RT-DETR)

**Proyecto:** Detección Automática de Fracturas Óseas en Radiografías  
**Materia:** Visión por Computadora II – CEIA/FIUBA  
**Autores:** Lucia T. Capon Paul · Cesar Orellana · Leandro Britez

---

## Objetivos
- Fine-tuning de RT-DETR (Real-Time DEtection TRansformer) como modelo alternativo.
- Mantener la misma estructura experimental que el notebook de YOLO para comparabilidad.
- **RT-DETR** fue elegido porque: (1) es un detector transformer de tiempo real, (2) también disponible en `ultralytics`, y (3) ofrece una comparación arquitectónica interesante frente al paradigma anchor-free de YOLO.

> **Nota:** Si el equipo decide usar otro modelo (ej. Faster R-CNN), reemplazar esta sección.

In [ ]:
import sys
sys.path.insert(0, '..')

from ultralytics import RTDETR
import yaml
from pathlib import Path
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

## 1. Configuración del experimento

In [ ]:
DATA_YAML = Path('../data/bone-fracture-detection-daoon-1/data.yaml')
assert DATA_YAML.exists(), 'Dataset no encontrado. Ver data/README.md'

with open('../configs/model2.yaml') as f:
    cfg = yaml.safe_load(f)

print('Configuración RT-DETR:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

## 2. Carga del modelo preentrenado

In [ ]:
model = RTDETR(cfg['model_weights'])
print(f'Modelo cargado: {cfg["model_weights"]}')

## 3. Fine-tuning

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=cfg['epochs'],
    imgsz=cfg['imgsz'],
    batch=cfg['batch'],
    lr0=cfg['lr0'],
    lrf=cfg['lrf'],
    weight_decay=cfg['weight_decay'],
    project='../results',
    name='rtdetr_fracture',
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
)

print('Entrenamiento finalizado.')
print(f'Mejor checkpoint: {results.save_dir}/weights/best.pt')

## 4. Evaluación rápida en validación

In [ ]:
val_results = model.val(data=str(DATA_YAML), split='val')
print(f'mAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'Precision:    {val_results.box.mp:.4f}')
print(f'Recall:       {val_results.box.mr:.4f}')

---
## Notas del experimento

*(Completar tras el entrenamiento)*

- **Variante de modelo usada:** ...
- **Épocas entrenadas / early stopping:** ...
- **mAP@0.5 final:** ...
- **Diferencias observadas vs YOLO:** ...